# 03b — Multi-frequency MIDAS (weekly target, daily + monthly regressors)

Mixed-Data Sampling at its **canonical** use: a low-frequency target regressed on
*higher*-frequency predictors, with parsimonious lag-weight polynomials compressing many
lags into a few shape parameters. This forecasts the **weekly** silver log-return (W-FRI)
from **two** sampling frequencies at once:

- **Daily block** — the 6 cross-asset daily returns (gold, USD, copper, S&P500, VIX, oil),
  each embedded over the last **K = 20 trading-day lags** and Beta/Almon-weighted. This
  **replaces** the weekly cross-asset EXOG lags of `03_midas` — a weekly lag is just the sum
  of its daily window, so the daily resolution is strictly richer.
- **Monthly block** — the publication-lagged monthly macro of `03_midas` (CPI, fed funds,
  industrial production, M2), 3 monthly lags each, same weighting.
- **Weekly base** — silver's own autocorrelation lags (`silver_lag1/2/3`).

Why bother: 6 × 20 = 120 daily lags make **restricted** weighting (Beta/Almon) *mandatory* —
U-MIDAS would estimate 120 free coefficients on ~330 weekly observations and overfit. So this
is the one place the weight polynomial is genuinely load-bearing (120 lags → 12 params).

Everything else mirrors the other weekly notebooks: same split, walk-forward (expanding, refit
every 4 weeks), the `eval_utils` §3a benchmarks (Naïve + Drift floor), and the DM-vs-Drift /
OOS R² / PT battery. **Honest expectation:** like every model here, this is not expected to beat
the drift — the value is a *stronger* semi-strong-efficiency null (public information at its
native daily resolution, optimally weighted, still doesn't forecast the weekly return).


## Setup

In [ ]:
suppressPackageStartupMessages({
  library(midasr)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(zoo)
})

# ── Evaluation metrics (mirror src/eval_utils.py exactly) ────────────────────
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p),      na.rm = TRUE)
da   <- function(a, p) mean(sign(a) == sign(p), na.rm = TRUE)
wda  <- function(a, p) {
  ok <- sign(a) == sign(p)
  sum(abs(a) * ok, na.rm = TRUE) / sum(abs(a), na.rm = TRUE)
}
eval_model <- function(name, actual, pred) {
  cat(sprintf("%-30s  RMSE=%.6f  MAE=%.6f  DA=%.3f  WDA=%.3f\n",
              name, rmse(actual, pred), mae(actual, pred),
              da(actual, pred), wda(actual, pred)))
  data.frame(model = name,
             rmse = rmse(actual, pred), mae = mae(actual, pred),
             dir_acc = da(actual, pred), wda = wda(actual, pred),
             stringsAsFactors = FALSE)
}

# Diebold-Mariano (Newey-West lag-1) — mirrors eval_utils.diebold_mariano.
#   loss = "se" squared-error (headline) | "ae" absolute-error (heavy-tail robustness).
#   Negative DM => pred1 more accurate. Returns list(dm, p) invisibly.
diebold_mariano <- function(actual, pred1, pred2, name1, name2, loss = "se",
                            verbose = TRUE) {
  mask <- !is.na(pred1) & !is.na(pred2) & !is.na(actual)
  a <- actual[mask]; p1 <- pred1[mask]; p2 <- pred2[mask]
  if (loss == "ae")      { e1 <- abs(a - p1);  e2 <- abs(a - p2) }
  else if (loss == "se") { e1 <- (a - p1)^2;   e2 <- (a - p2)^2  }
  else stop("loss must be 'se' (squared) or 'ae' (absolute)")
  d <- e1 - e2
  n  <- length(d); d_bar <- mean(d)
  gamma0 <- var(d, na.rm = TRUE)                       # ddof=1, matches numpy var(ddof=1)
  gamma1 <- if (n > 1) cov(d[-n], d[-1]) else 0        # NW lag-1 autocovariance
  var_d  <- (gamma0 + 2 * gamma1) / n
  if (is.na(var_d) || var_d <= 0) {
    if (verbose) cat(sprintf("%-26s vs %-26s [%s]  variance non-positive, skipping\n",
                             name1, name2, loss))
    return(invisible(list(dm = NA_real_, p = NA_real_)))
  }
  dm <- d_bar / sqrt(var_d)
  pv <- 2 * (1 - pnorm(abs(dm)))
  if (verbose) {
    sig <- if (pv < 0.001) "***" else if (pv < 0.01) "**" else if (pv < 0.05) "*" else "(ns)"
    cat(sprintf("%-26s vs %-26s [%s]  DM=%+.3f  p=%.3f  %s\n",
                name1, name2, loss, dm, pv, sig))
  }
  invisible(list(dm = dm, p = pv))
}

# Campbell-Thompson (2008) out-of-sample R^2 vs a benchmark (the drift).
#   >0 => beats the random-walk-with-drift OOS; <0 (usual for returns) => worse.
oos_r2 <- function(actual, pred, benchmark) {
  m <- !is.na(actual) & !is.na(pred) & !is.na(benchmark)
  a <- actual[m]; p <- pred[m]; b <- benchmark[m]
  sse_b <- sum((a - b)^2)
  if (sse_b == 0) NA_real_ else 1 - sum((a - p)^2) / sse_b
}

# Pesaran-Timmermann (1992) sign-predictability test — mirrors
# eval_utils.pesaran_timmermann. Two-sided, base-rate aware. PT/p = NA when
# degenerate (a constant-sign forecast carries no directional information).
pesaran_timmermann <- function(actual, pred) {
  m <- !is.na(actual) & !is.na(pred)
  a <- actual[m]; p <- pred[m]
  n  <- length(a)
  dy <- as.numeric(a > 0); dx <- as.numeric(p > 0)
  hit   <- mean(dy == dx)                              # DA (sign hit rate)
  py <- mean(dy); px <- mean(dx)
  Pstar <- py * px + (1 - py) * (1 - px)               # hit rate under independence
  var_hit   <- Pstar * (1 - Pstar) / n
  var_pstar <- ((2 * py - 1)^2 * px * (1 - px) / n
                + (2 * px - 1)^2 * py * (1 - py) / n
                + 4 * py * px * (1 - py) * (1 - px) / n^2)
  denom <- var_hit - var_pstar
  if (px %in% c(0, 1) || py %in% c(0, 1) || is.na(denom) || denom <= 0)
    return(list(n = n, DA = hit, DA_indep = Pstar, PT = NA_real_, p = NA_real_))
  pt   <- (hit - Pstar) / sqrt(denom)
  pval <- 2 * (1 - pnorm(abs(pt)))                     # two-sided: any sign dependence
  list(n = n, DA = hit, DA_indep = Pstar, PT = pt, p = pval)
}

cat("midasr:", as.character(packageVersion("midasr")), "\n")


## 1. Load & aggregate to weekly

Same input files as `04_random_forest.ipynb` / `05_xgboost.ipynb`. Daily returns
are summed to W-FRI (sums = log-return additivity); monthly macro is joined later
at its native frequency.

In [ ]:
train <- read.csv("../../data/processed/train.csv", row.names = 1)
val   <- read.csv("../../data/processed/val.csv",   row.names = 1)
test  <- read.csv("../../data/processed/test.csv",  row.names = 1)

TARGET       <- "silver_return"
EXOG_RETURNS <- c("gold_return", "usd_return", "copper_return",
                  "sp500_return", "vix_return", "oil_return")
EXOG_LEVELS  <- c("gs_ratio_z")       # level features — separate ablation group
EXOG         <- c(EXOG_RETURNS, EXOG_LEVELS)

# Friday-end weekly aggregation (matches pandas .resample('W-FRI').sum()).
# Returns are summed (additive log-returns); levels take the last value (Friday close).
to_weekly <- function(df) {
  df$Date <- as.Date(rownames(df))
  df$week_end <- df$Date + ((5 - as.integer(format(df$Date, "%u"))) %% 7)
  ret_cols <- intersect(c(TARGET, EXOG_RETURNS), names(df))
  lvl_cols <- intersect(EXOG_LEVELS, names(df))

  agg <- df %>%
    group_by(week_end) %>%
    summarise(
      across(all_of(ret_cols), ~ sum(.x, na.rm = TRUE)),
      across(all_of(lvl_cols), ~ {
        # take the last non-NA value within the week
        non_na <- .x[!is.na(.x)]
        if (length(non_na) == 0) NA_real_ else tail(non_na, 1)
      }),
      .groups = "drop"
    ) %>%
    rename(Date = week_end)
  agg
}

train_w <- to_weekly(train)
val_w   <- to_weekly(val)
test_w  <- to_weekly(test)

all_w   <- bind_rows(train_w, val_w, test_w) %>% arrange(Date)
n_train <- nrow(train_w) + nrow(val_w)

cat(sprintf("Weekly obs — train+val: %d, test: %d\n", n_train, nrow(test_w)))
cat("Cols:", paste(names(all_w), collapse = ", "), "\n")

## 2. Base predictors — silver autocorrelation lags

The base is silver's own 3 weekly return lags (`silver_lag1/2/3`) — the AR term / weak-form
benchmark. Cross-asset predictors are **not** here as weekly lags; they enter at daily
resolution through the MIDAS block in §3. (`02_features`-style weekly cross-asset lags and the
`gs_ratio_z` level are still built below so the `+GS` rung exists, but the 6 weekly cross-asset
lags are deliberately unused.)


In [ ]:
# Silver lags 1-3
for (k in 1:3) all_w[[paste0("silver_lag", k)]] <- dplyr::lag(all_w$silver_return, k)

# 1-week lag of each cross-asset return
for (v in EXOG_RETURNS) {
  lag_name <- sub("_return", "_lag1", v)
  if (v %in% names(all_w)) all_w[[lag_name]] <- dplyr::lag(all_w[[v]], 1)
}

# 1-week lag of each level feature (gs_ratio_z) — separate group for ablation
for (v in EXOG_LEVELS) {
  if (v %in% names(all_w)) all_w[[paste0(v, "_lag1")]] <- dplyr::lag(all_w[[v]], 1)
}

EXOG_LAGS <- c("silver_lag1", "silver_lag2", "silver_lag3",
               "gold_lag1", "usd_lag1", "copper_lag1",
               "sp500_lag1", "vix_lag1", "oil_lag1")
EXOG_LAGS <- intersect(EXOG_LAGS, names(all_w))

GS_LAGS <- intersect(paste0(EXOG_LEVELS, "_lag1"), names(all_w))

cat("EXOG base regressors (", length(EXOG_LAGS), "):", paste(EXOG_LAGS, collapse = ", "), "\n")
cat("GS group (", length(GS_LAGS), "):", paste(GS_LAGS, collapse = ", "), "\n")

## 3. Daily high-frequency block (MIDAS predictors)

For each weekly Friday `week_end`, take the **K = 20 most recent daily returns** of each
cross-asset series observable by the position Friday $F_{t-1}$ — daily date `< week_end − 6`
(the same no-look-ahead rule as the monthly block, in trading-day space so holidays never slip a
day across the week boundary). Daily price returns are known at the close, so no publication lag
applies.

Each series becomes an (N × 20) matrix; the weight polynomial in §6 collapses its 20 lags into
one weighted scalar per week. The most-recent lag (`*_dlag1`) is the $F_{t-1}$ close — the
position-day return, known at decision time. None of the **target-week** daily returns enter, so
there is no overlap with the weekly target (which is built from those same days).


In [ ]:
# ── Daily high-frequency lag matrices ────────────────────────────────────────
daily <- rbind(train, val, test)
daily$Date <- as.Date(rownames(daily))
daily <- daily[order(daily$Date), ]

DAILY_VARS <- c("gold_return", "usd_return", "copper_return",
                "sp500_return", "vix_return", "oil_return")
stopifnot(all(DAILY_VARS %in% names(daily)))      # error loud if a series is missing
K_DAILY <- 20L                                    # trading-day lags compressed by the weights

# K most recent daily returns observable by F_{t-1} (date < week_end - 6); col 1 = most recent.
build_weekly_daily_lags <- function(weekly_dates, daily_df, var, k) {
  d_dates <- daily_df$Date
  d_vals  <- daily_df[[var]]
  obs     <- !is.na(d_vals)
  mat <- matrix(NA_real_, nrow = length(weekly_dates), ncol = k)
  for (i in seq_along(weekly_dates)) {
    keep <- obs & d_dates < weekly_dates[i] - 6   # observable by the position Friday F_{t-1}
    past <- d_vals[keep]
    if (length(past) >= k) mat[i, ] <- rev(tail(past, k))
  }
  colnames(mat) <- paste0(sub("_return", "", var), "_dlag", seq_len(k))
  mat
}

daily_lags <- lapply(DAILY_VARS, function(v)
  build_weekly_daily_lags(all_w$Date, daily, v, K_DAILY))
names(daily_lags) <- sub("_return", "", DAILY_VARS)

cat(sprintf("Daily lag matrices: %d series x %d trading-day lags, aligned to %d weeks\n",
            length(daily_lags), K_DAILY, nrow(all_w)))


### Look-ahead audit — daily block

Independently confirm the most-recent daily lag of every week is dated on or before the position
Friday $F_{t-1}$ = `week_end − 7`. Slack ≥ 0 for every week (0 when $F_{t-1}$ is a trading day).


In [ ]:
position_fri <- all_w$Date - 7
for (v in names(daily_lags)) {
  src_var   <- paste0(v, "_return")
  min_slack <- Inf
  for (i in seq_len(nrow(all_w))) {
    keep <- !is.na(daily[[src_var]]) & daily$Date < all_w$Date[i] - 6
    if (any(keep))
      min_slack <- min(min_slack, as.numeric(position_fri[i] - max(daily$Date[keep])))
  }
  if (is.finite(min_slack) && min_slack < 0)
    stop(sprintf("Look-ahead detected in daily '%s' (slack %d days)", v, as.integer(min_slack)))
  cat(sprintf("  %-8s OK  (most recent daily lag <= F_{t-1}; min slack %d days)\n",
              v, as.integer(min_slack)))
}
cat("\nDaily look-ahead audit passed — no daily return enters before F_{t-1}.\n")


## 4. Monthly macro block — the second frequency (MIDAS predictors)

This is the **mixed-frequency-specific** part of the notebook. We build, for
each weekly Friday, a vector of the **3 most recent monthly observations** of
each macro variable that were *observable* by that date. These vectors are
stacked into (N × 3) matrices — one per macro variable — and later fed through
the weight polynomial in §5.

### Lag-matrix structure

For each weekly date `t` and each variable `v`, row `t` of `macro_lags[[v]]`
looks like:

```
[ v_at_month_just_before_t,    # lag 1 = most recent
  v_at_month_before_that,      # lag 2
  v_at_month_before_that ]     # lag 3
```

Later, `w_1·v_lag1 + w_2·v_lag2 + w_3·v_lag3` collapses these three numbers
into a single scalar per week. The weights `w_1, w_2, w_3` are what MIDAS
learns (via the Beta/Almon polynomial) or estimates as free coefficients (U-MIDAS).

### Timing — real-time observability (no look-ahead)

This is the subtle part. `monthly_macro.csv` stamps each monthly value on the
**1st of its reference month** — e.g. March CPI is stored at `2026-03-01`. But
March CPI is not *published* until ~April 12. A naïve filter of
`macro$Date < week_end` would let a model forecasting an early-March week use
the March CPI value roughly six weeks before it existed.

Two corrections are applied so the macro block only ever uses genuinely
observable data:

1. **Publication lag.** Each variable gets a conservative release delay added to
   its month-start stamp, producing an `available_date`:

   | Variable | FRED series | Released | Lag from month-1st stamp |
   |---|---|---|---|
   | `cpi` | CPIAUCSL | ~10–15th of next month | +46 days |
   | `ind_prod` | INDPRO | ~15–17th of next month | +48 days |
   | `fed_funds` | FEDFUNDS | start of next month | +35 days |
   | `m2` | M2SL | ~1mo after month-end | +30 days |

   The lags are deliberately generous — it is safe to treat data as available
   slightly *later* than reality, never earlier.

2. **Position-time alignment.** A row predicting the return for week
   $F_{t-1} \to F_t$ is traded at the *prior* Friday $F_{t-1}$, and every EXOG
   feature is already lagged to $F_{t-1}$. The macro block is held to the same
   instant: the filter is `available_date < week_end - 6`, i.e. observable on or
   before $F_{t-1}$ — not merely before the current Friday.

The audit cell below independently re-checks that no macro value enters a row
before its `available_date`.

**Limitation.** This controls *publication timing* but not *data revisions*:
FRED serves the latest revised figures, which differ from the values first
released. A fully real-time treatment would need ALFRED vintages; that is left
as a stated robustness extension.

### Why 3 lags?

With $K = 3$ we span roughly the last quarter of macro history per weekly
observation — long enough to capture the typical 2-3 month publication delay
+ leading-indicator effect, short enough not to dilute the signal across stale
old releases. Increasing $K$ wouldn't add parameters under Beta/Almon (still 2
shape params each) but would burn early training data while the lag matrix
warms up.

In [ ]:
macro <- read.csv("../../data/raw/monthly_macro.csv", row.names = 1)
macro$Date <- as.Date(rownames(macro))

MACRO_VARS   <- c("cpi", "fed_funds", "ind_prod", "m2")   # m2, not real_rates (real rate is daily via DFII10)
stopifnot(all(MACRO_VARS %in% names(macro)))              # error loud if a column is missing
N_MACRO_LAGS <- 3

# Conservative publication lag (days) added to each variable's month-1st stamp,
# turning a reference-month date into a real-time `available_date`. See §3 md.
MACRO_AVAIL_LAG <- c(cpi = 46, ind_prod = 48, fed_funds = 35, m2 = 30)  # m2: H.6 ~1mo after month-end (matches 02_features)

# For each weekly Friday, take the n_lags most recent monthly observations whose
# `available_date` falls on or before the prior Friday (week_end - 7) — the
# position-taking date — so macro is observed at the same instant as EXOG.
build_weekly_macro_lags <- function(weekly_dates, macro_df, var, n_lags, lag_days) {
  avail <- macro_df$Date + lag_days                  # real-time availability date
  obs   <- !is.na(macro_df[[var]])                   # rows that actually carry a value
  mat   <- matrix(NA_real_, nrow = length(weekly_dates), ncol = n_lags)
  for (i in seq_along(weekly_dates)) {
    keep <- obs & avail < weekly_dates[i] - 6        # observable by F_{t-1}
    past <- macro_df[[var]][keep]
    if (length(past) >= n_lags) mat[i, ] <- rev(tail(past, n_lags))
  }
  colnames(mat) <- paste0(var, "_mlag", seq_len(n_lags))
  mat
}

macro_lags <- lapply(MACRO_VARS, function(v)
  build_weekly_macro_lags(all_w$Date, macro, v, N_MACRO_LAGS, MACRO_AVAIL_LAG[[v]]))
names(macro_lags) <- MACRO_VARS

cat("Macro lag matrices built for:", paste(MACRO_VARS, collapse = ", "),
    "(", N_MACRO_LAGS, "monthly lags each)\n")
cat("Publication lags (days):  ",
    paste(sprintf("%s=%d", names(MACRO_AVAIL_LAG), MACRO_AVAIL_LAG), collapse = "   "), "\n")

### Look-ahead audit

Independently re-derives the monthly observation behind every `*_mlag1` and
asserts (via `stop()` on failure) that its `available_date` never postdates the
position-taking Friday $F_{t-1}$. The spot check then shows, for the most recent
weeks, which reference month feeds `cpi_mlag1` and when it became observable —
so the guarantee is visible, not just claimed.

In [ ]:
# Independently re-derive the macro observation behind each *_mlag1 and confirm
# its availability date never postdates the position Friday (F_{t-1} = week - 7).
position_fri <- all_w$Date - 7

for (v in MACRO_VARS) {
  avail_v   <- macro$Date + MACRO_AVAIL_LAG[[v]]
  obs_v     <- !is.na(macro[[v]])
  min_slack <- Inf
  for (i in seq_len(nrow(all_w))) {
    keep <- obs_v & avail_v < all_w$Date[i] - 6
    if (any(keep))
      min_slack <- min(min_slack, as.numeric(position_fri[i] - max(avail_v[keep])))
  }
  if (is.finite(min_slack) && min_slack < 0)
    stop(sprintf("Look-ahead detected in '%s' (slack %d days)", v, as.integer(min_slack)))
  cat(sprintf("  %-11s OK  (tightest week: value available %d days before the position Friday)\n",
              v, as.integer(min_slack)))
}

# Human-readable spot check — last 4 weeks with a cpi value, cpi_mlag1 source.
cat("\nSpot check — cpi_mlag1 source month vs forecast week:\n")
recent <- tail(which(!is.na(macro_lags[["cpi"]][, 1])), 4)
for (i in recent) {
  src <- macro$Date[!is.na(macro$cpi) &
                    macro$Date + MACRO_AVAIL_LAG[["cpi"]] < all_w$Date[i] - 6]
  cat(sprintf("  forecast week %s  ->  cpi from %s  (available ~%s, position Fri %s)\n",
              all_w$Date[i], format(max(src), "%Y-%m"),
              max(src) + MACRO_AVAIL_LAG[["cpi"]], all_w$Date[i] - 7))
}
cat("\nLook-ahead audit passed — no macro value is used before it was observable.\n")

## 5. Weekly sentiment (lagged 1 week)

Aggregate daily Reddit + GDELT-news sentiment to W-FRI mean, forward-fill any gaps,
lag 1 week. Matches the sentiment treatment in `04_random_forest.ipynb` etc.

In [ ]:
sent_path <- "../../data/processed/daily_sentiment.csv"
sentiment_available <- file.exists(sent_path)

ffill_zero <- function(x) { x <- zoo::na.locf(x, na.rm = FALSE); x[is.na(x)] <- 0; x }

if (sentiment_available) {
  sent <- read.csv(sent_path, row.names = 1)
  sent$Date <- as.Date(rownames(sent))
  sent$week_end <- sent$Date + ((5 - as.integer(format(sent$Date, "%u"))) %% 7)

  sent_cols <- intersect(c("reddit_sentiment", "news_sentiment"), names(sent))
  sent_w <- sent %>%
    group_by(week_end) %>%
    summarise(across(all_of(sent_cols), ~ mean(.x, na.rm = TRUE)), .groups = "drop") %>%
    rename(Date = week_end)

  aligned <- data.frame(Date = all_w$Date) %>% left_join(sent_w, by = "Date")

  if ("reddit_sentiment" %in% names(aligned)) {
    all_w$reddit_lag1 <- dplyr::lag(ffill_zero(aligned$reddit_sentiment), 1)
    all_w$reddit_lag1[is.na(all_w$reddit_lag1)] <- 0
  }
  if ("news_sentiment" %in% names(aligned)) {
    all_w$news_lag1 <- dplyr::lag(ffill_zero(aligned$news_sentiment), 1)
    all_w$news_lag1[is.na(all_w$news_lag1)] <- 0
  }
  cat("Sentiment merged. Columns added:",
      paste(intersect(c("reddit_lag1", "news_lag1"), names(all_w)), collapse = ", "), "\n")
} else {
  cat("daily_sentiment.csv not found — sentiment variants will be skipped.\n")
}


## 6. MIDAS weight functions and fitters

The estimation engine, reused unchanged from `03_midas`. `nbeta_w` / `nealmon_w` map K lag
indices to weights summing to 1 via 2 shape parameters; `fit_with_midas` jointly fits the shape
parameters (L-BFGS) and the OLS coefficients; `predict_with_midas` reuses the learned weights.
The fitter is **generic over a list of lag matrices**, so the daily block and the monthly block
are passed together — each series gets its own 2-parameter weight curve. U-MIDAS is intentionally
omitted: 120 daily lags as free coefficients would overfit.


In [ ]:
# Beta polynomial weights -> K weights summing to 1.
#   k     : integer lag indices (1..K)
#   theta : 2-vector (theta_1, theta_2), both > 0
nbeta_w <- function(k, theta) {
  th <- pmax(theta, 0.1)          # Beta needs strictly positive shape params
  x  <- k / (max(k) + 1)          # rescale lag indices to (0, 1)
  w  <- x^(th[1] - 1) * (1 - x)^(th[2] - 1)
  w  <- pmax(w, 1e-10)            # avoid pathological 0 weights (division later)
  w / sum(w)                      # normalise -> sum w_k = 1
}

In [ ]:
# Normalised exponential Almon weights — sign-unrestricted shape parameters.
nealmon_w <- function(k, theta) {
  w <- exp(theta[1] * k + theta[2] * k^2)
  w / sum(w)
}

In [ ]:
# Restricted MIDAS (Beta or Almon) — joint fit.
# L-BFGS-B searches the shape params theta; OLS coefs are closed-form per theta.
fit_with_midas <- function(y, base_X, macro_list, weight_fn = nbeta_w,
                            start = NULL, lower = NULL) {
  vars   <- names(macro_list)
  nmacro <- length(vars)
  k_list <- lapply(macro_list, function(m) seq_len(ncol(m)))   # lag indices per var

  # Default starting θ values: Beta (1, 5) ≈ monotone decay; Almon (0, 0) ≈ flat.
  if (is.null(start)) start <- rep(c(1, 5), nmacro)
  if (is.null(lower)) lower <- rep(0.1, 2 * nmacro)

  # Inner objective: SSR(θ) given fresh OLS coefficients for this θ.
  obj <- function(theta_vec) {
    cols <- vector("list", nmacro)
    for (j in seq_len(nmacro)) {
      th <- theta_vec[(2 * j - 1):(2 * j)]                # 2 params per macro var
      cols[[j]] <- macro_list[[j]] %*% weight_fn(k_list[[j]], th)
    }
    midas_X <- do.call(cbind, cols)                       # (N × n_macro) weighted scalars
    X <- cbind(1, as.matrix(base_X), midas_X)             # [intercept | EXOG | macro-weighted]
    b <- tryCatch(solve(crossprod(X), crossprod(X, y)),   # OLS β̂ = (X'X)⁻¹ X'y
                  error = function(e) NULL)
    if (is.null(b)) return(1e9)                           # singular → huge penalty
    sum((y - X %*% b)^2)                                  # SSR
  }

  # L-BFGS-B: gradient-free (numerical), with box constraints `lower`.
  # factr = 1e7 keeps the convergence tolerance lenient → faster, since we don't
  # need super-tight optima for forecasting.
  opt <- optim(start, obj, method = "L-BFGS-B", lower = lower,
               control = list(maxit = 1000, factr = 1e7))

  # Reconstruct best weights & final OLS coefficients with the converged θ.
  theta_vec <- opt$par
  weights   <- list()
  cols_tr   <- vector("list", nmacro)
  for (j in seq_len(nmacro)) {
    th <- theta_vec[(2 * j - 1):(2 * j)]
    w  <- weight_fn(k_list[[j]], th)
    weights[[vars[j]]] <- w
    cols_tr[[j]]       <- macro_list[[j]] %*% w
  }
  midas_tr <- do.call(cbind, cols_tr); colnames(midas_tr) <- paste0(vars, "_midas")
  X_tr <- cbind(intercept = 1, as.matrix(base_X), midas_tr)
  b    <- as.vector(solve(crossprod(X_tr), crossprod(X_tr, y)))

  list(spec = "restricted", weights = weights, coefs = b, theta = theta_vec,
       converged = (opt$convergence == 0))
}

In [ ]:
# Predict with a fitted restricted-MIDAS model: reuse learned weights + coefs.
predict_with_midas <- function(fit, base_X, macro_list) {
  vars <- names(fit$weights)
  cols <- lapply(vars, function(v) macro_list[[v]] %*% fit$weights[[v]])
  midas_te <- do.call(cbind, cols); colnames(midas_te) <- paste0(vars, "_midas")
  X_te <- cbind(intercept = 1, as.matrix(base_X), midas_te)
  as.vector(X_te %*% fit$coefs)
}

## 7. Two-stage protocol — spec selection, then walk-forward ladder

**Stage 1** picks the restricted weight family (**Beta** vs **Almon**) by validation WDA (fit on
train-only, scored on 2022 val), with the full multi-frequency list (6 daily + 4 monthly) held
fixed. U-MIDAS is not a candidate (it would overfit the 120 daily lags).

**Stage 2** runs the ablation ladder with a **walk-forward expanding window**, refitting every
4 weeks (RF/XGB cadence). The **daily cross-asset block is in the base of every variant** (it
replaces the weekly EXOG); the **monthly macro block** is toggled by the `+Macro` rungs; `+GS`
and the sentiment rungs add plain weekly regressors.

| Variant | silver AR | daily block | monthly macro | extras |
|---|---|---|---|---|
| Naïve / Drift | — | — | — | benchmarks, no fit |
| `EXOG-d` | ✓ | ✓ | | |
| `EXOG-d+Macro` | ✓ | ✓ | ✓ | the multi-frequency headline |
| `EXOG-d+GS` | ✓ | ✓ | | gs_ratio_z |
| `EXOG-d+Reddit / +News / +Reddit+News` | ✓ | ✓ | | sentiment |
| `EXOG-d+Macro+Sentiment` | ✓ | ✓ | ✓ | sentiment |
| `EXOG-d+ALL` | ✓ | ✓ | ✓ | gs + sentiment |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STAGE 1 — pick the restricted weight family (Beta vs Almon) by val WDA.
#   Combined multi-frequency list (6 daily + 4 monthly) held fixed; weights vary.
#   U-MIDAS excluded: 6 x 20 daily lags as free coefficients would overfit.
# ═══════════════════════════════════════════════════════════════════════════
AR_LAGS     <- intersect(c("silver_lag1", "silver_lag2", "silver_lag3"), names(all_w))
GS_LAGS     <- intersect(paste0(EXOG_LEVELS, "_lag1"), names(all_w))
mf_list_all <- c(daily_lags, macro_lags)          # daily block + monthly macro block

make_mask <- function(y, b, m_list)
  complete.cases(b) & !is.na(y) & Reduce(`&`, lapply(m_list, complete.cases))

n_train_only <- nrow(train_w)
y_all_tr     <- all_w$silver_return[1:n_train]
base_all_tr  <- all_w[1:n_train, AR_LAGS, drop = FALSE]
mf_all_tr    <- lapply(mf_list_all, function(M) M[1:n_train, , drop = FALSE])

idx_tr1 <- 1:n_train_only
y_tr1 <- y_all_tr[idx_tr1]; b_tr1 <- base_all_tr[idx_tr1, , drop = FALSE]
m_tr1 <- lapply(mf_all_tr, function(M) M[idx_tr1, , drop = FALSE])
mk    <- make_mask(y_tr1, b_tr1, m_tr1)
y_tr1 <- y_tr1[mk]; b_tr1 <- b_tr1[mk, , drop = FALSE]
m_tr1 <- lapply(m_tr1, function(M) M[mk, , drop = FALSE])

idx_v <- (n_train_only + 1):n_train
y_v <- y_all_tr[idx_v]; b_v <- base_all_tr[idx_v, , drop = FALSE]
m_v <- lapply(mf_all_tr, function(M) M[idx_v, , drop = FALSE])
mkv <- make_mask(y_v, b_v, m_v)
y_v <- y_v[mkv]; b_v <- b_v[mkv, , drop = FALSE]
m_v <- lapply(m_v, function(M) M[mkv, , drop = FALSE])

cat(sprintf("Stage 1 — train-only: %d obs   val: %d obs   (%d MIDAS series: %d daily + %d monthly)\n",
            length(y_tr1), length(y_v), length(mf_list_all),
            length(daily_lags), length(macro_lags)))

kk <- length(m_tr1)
fit_b_s1 <- fit_with_midas(y_tr1, b_tr1, m_tr1, weight_fn = nbeta_w,
                           start = rep(c(1, 5), kk), lower = rep( 0.1, 2 * kk))
fit_a_s1 <- fit_with_midas(y_tr1, b_tr1, m_tr1, weight_fn = nealmon_w,
                           start = rep(c(0, 0), kk), lower = rep(-5.0, 2 * kk))
pred_b_v <- predict_with_midas(fit_b_s1, b_v, m_v)
pred_a_v <- predict_with_midas(fit_a_s1, b_v, m_v)

cat("\nStage 1 — val scores:\n")
stage1    <- rbind(eval_model("Beta-MIDAS",  y_v, pred_b_v),
                   eval_model("Almon-MIDAS", y_v, pred_a_v))
best_spec <- stage1$model[which.max(stage1$wda)]
cat("\n=> Winner by val WDA:", best_spec, "\n")
write.csv(stage1, "../../data/processed/midas_daily_stage1_specs.csv", row.names = FALSE)

# ═══════════════════════════════════════════════════════════════════════════
# STAGE 2 — walk-forward ablation ladder (expanding, refit every 4 weeks).
# ═══════════════════════════════════════════════════════════════════════════
RETRAIN_EVERY <- 4L

mask_all <- make_mask(all_w$silver_return, all_w[, AR_LAGS, drop = FALSE], mf_list_all)
y_F      <- all_w$silver_return[mask_all]
ar_F     <- all_w[mask_all, AR_LAGS, drop = FALSE]
gs_F     <- all_w[mask_all, GS_LAGS, drop = FALSE]
daily_F  <- lapply(daily_lags, function(M) M[mask_all, , drop = FALSE])
macro_F  <- lapply(macro_lags, function(M) M[mask_all, , drop = FALSE])
dates_F  <- all_w$Date[mask_all]
test_idx <- which(which(mask_all) > n_train)
y_te     <- y_F[test_idx]
dates_te <- dates_F[test_idx]
if (sentiment_available) { r_F <- all_w$reddit_lag1[mask_all]; n_F <- all_w$news_lag1[mask_all] }

cat(sprintf("\nStage 2 — masked obs: %d   test obs: %d   refit every %d weeks\n",
            length(y_F), length(test_idx), RETRAIN_EVERY))

fit_restricted <- function(y, base_X, midas_list) {
  kk <- length(midas_list)
  if (best_spec == "Beta-MIDAS")
    fit_with_midas(y, base_X, midas_list, weight_fn = nbeta_w,
                   start = rep(c(1, 5), kk), lower = rep( 0.1, 2 * kk))
  else
    fit_with_midas(y, base_X, midas_list, weight_fn = nealmon_w,
                   start = rep(c(0, 0), kk), lower = rep(-5.0, 2 * kk))
}

# Walk-forward: midas_F = lag-matrix list for this variant (always incl. the daily
# block); base_X_full = silver AR (+ any plain weekly extras).
walk_forward <- function(base_X_full, midas_F) {
  preds <- rep(NA_real_, length(test_idx)); fit <- NULL
  for (j in seq_along(test_idx)) {
    i <- test_idx[j]
    if (j == 1L || ((j - 1L) %% RETRAIN_EVERY) == 0L) {
      tr  <- seq_len(i - 1L)
      fit <- fit_restricted(y_F[tr], base_X_full[tr, , drop = FALSE],
                            lapply(midas_F, function(M) M[tr, , drop = FALSE]))
    }
    preds[j] <- predict_with_midas(fit, base_X_full[i, , drop = FALSE],
                                   lapply(midas_F, function(M) M[i, , drop = FALSE]))
  }
  list(pred = preds, fit = fit)
}

all_results <- list(); all_preds <- list()
run_variant <- function(label, base_X_full, midas_F) {
  wf <- walk_forward(base_X_full, midas_F)
  all_preds[[label]]   <<- wf$pred
  all_results[[label]] <<- eval_model(label, y_te, wf$pred)
  invisible(wf$fit)
}

# Naive baseline (Drift added in the next cell)
naive_pred <- c(NA, y_te[-length(y_te)]); nz <- !is.na(naive_pred)
all_preds$"Naive (t-1 week)"   <- naive_pred
all_results$"Naive (t-1 week)" <- eval_model("Naive (t-1 week)", y_te[nz], naive_pred[nz])

# ── Ladder — the daily block is in the base of every fitted variant ──────────
run_variant("EXOG-d",                 ar_F,                                            daily_F)
fit_em <- run_variant("EXOG-d+Macro", ar_F,                                            c(daily_F, macro_F))
if (length(GS_LAGS) > 0)
  run_variant("EXOG-d+GS",            cbind(ar_F, gs_F),                               daily_F)
if (sentiment_available) {
  run_variant("EXOG-d+Reddit",        cbind(ar_F, reddit_lag1 = r_F),                  daily_F)
  run_variant("EXOG-d+News",          cbind(ar_F, news_lag1 = n_F),                    daily_F)
  run_variant("EXOG-d+Reddit+News",   cbind(ar_F, reddit_lag1 = r_F, news_lag1 = n_F), daily_F)
  run_variant("EXOG-d+Macro+Sentiment",
              cbind(ar_F, reddit_lag1 = r_F, news_lag1 = n_F),                         c(daily_F, macro_F))
  if (length(GS_LAGS) > 0)
    run_variant("EXOG-d+ALL",
                cbind(ar_F, gs_F, reddit_lag1 = r_F, news_lag1 = n_F),                 c(daily_F, macro_F))
}


In [ ]:
# ── Drift (prevailing mean) baseline ─────────────────────────────────────────
# Expanding historical mean of returns = random-walk-with-drift = ARIMA(0,0,0):
# the correct EMH floor for a return target (Welch-Goyal / Campbell-Thompson) and
# the always-up directional line (its sign is constant-positive). Walk-forward, so
# each test week sees only returns up to F_{t-1}.
drift_pred <- vapply(test_idx,
                     function(i) mean(y_F[seq_len(i - 1L)], na.rm = TRUE),
                     numeric(1))
all_preds$"Drift (prevailing mean)"   <- drift_pred
all_results$"Drift (prevailing mean)" <- eval_model("Drift (prevailing mean)", y_te, drift_pred)

# Lead the table with the two no-fit benchmarks, then the fitted variants.
.lead       <- c("Naive (t-1 week)", "Drift (prevailing mean)")
.ord        <- c(.lead, setdiff(names(all_results), .lead))
all_results <- all_results[.ord]
all_preds   <- all_preds[.ord]


## 8. Results table

RMSE / MAE / DA / WDA + Campbell-Thompson OOS R² (vs the Drift floor) per variant. RMSE is
primary; OOS R² > 0 would mean the variant beats the random-walk drift out of sample.


In [ ]:
metrics_df <- do.call(rbind, all_results)
rownames(metrics_df) <- NULL

# Campbell-Thompson OOS R^2 vs the Drift floor (effect size; >0 => beats the random walk).
drift_p <- all_preds[["Drift (prevailing mean)"]]
metrics_df$oos_r2 <- vapply(metrics_df$model,
  function(nm) oos_r2(y_te, all_preds[[nm]], drift_p), numeric(1))

write.csv(metrics_df, "../../data/processed/metrics_midas_daily_weekly.csv", row.names = FALSE)

cat(sprintf("%-30s %10s %10s %6s %6s %9s\n", "Model", "RMSE", "MAE", "DA", "WDA", "OOS_R2"))
cat(strrep("-", 78), "\n")
for (i in seq_len(nrow(metrics_df))) {
  r <- metrics_df[i, ]
  cat(sprintf("%-30s %10.6f %10.6f %6.3f %6.3f %+9.4f\n",
              r$model, r$rmse, r$mae, r$dir_acc, r$wda, r$oos_r2))
}


## 9. Fitted lag-weight profiles (`EXOG-d+Macro`)

The learned Beta/Almon weight curves per series. The 6 daily blocks span 20 trading-day lags
(how far back the daily signal is weighted); the 4 monthly blocks span 3 lags. Lag 1 = most recent.


In [ ]:
stopifnot(fit_em$spec == "restricted")
w_df <- do.call(rbind, lapply(names(fit_em$weights), function(v) {
  w <- fit_em$weights[[v]]
  data.frame(var = v, lag = seq_along(w), weight = w,
             freq = ifelse(length(w) > 5, "daily (20 lags)", "monthly (3 lags)"))
}))
ggplot(w_df, aes(x = lag, y = weight, colour = freq)) +
  geom_line() + geom_point(size = 1.1) +
  facet_wrap(~ var, scales = "free", ncol = 3) +
  scale_colour_manual(values = c("daily (20 lags)" = "steelblue",
                                 "monthly (3 lags)" = "#c0392b")) +
  labs(title = paste0("EXOG-d+Macro: fitted ", best_spec, " lag-weight profiles"),
       x = "Lag (1 = most recent)", y = "Weight", colour = NULL) +
  theme_minimal(base_size = 10) + theme(legend.position = "top")


## 10. Period breakdown — DA / WDA by calendar year

Best variant by WDA broken down across the standard `PERIODS` from `eval_utils`.
Same year buckets as RF / XGBoost / LSTM.

In [ ]:
# Pick best non-naive variant by WDA
candidates <- metrics_df[!metrics_df$model %in% c("Naive (t-1 week)", "Drift (prevailing mean)"), ]
best_row   <- candidates[which.max(candidates$wda), ]
best_name  <- best_row$model
best_pred  <- all_preds[[best_name]]
cat("Best variant by WDA:", best_name, "\n")

PERIODS <- list(
  "2023 (choppy)"     = c("2023-01-01", "2023-12-31"),
  "2024 (bull start)" = c("2024-01-01", "2024-12-31"),
  "2025 (bull run)"   = c("2025-01-01", "2025-12-31"),
  "2026 (YTD)"        = c("2026-01-01", "2026-12-31"),
  "── Full test ──"   = c("2023-01-01", "2026-12-31")
)

rows <- list()
for (label in names(PERIODS)) {
  bounds <- as.Date(PERIODS[[label]])
  mask   <- dates_te >= bounds[1] & dates_te <= bounds[2]
  a <- y_te[mask]; p <- best_pred[mask]
  ok <- !is.na(a) & !is.na(p)
  if (sum(ok) < 4) next
  da_v  <- mean(sign(a[ok]) == sign(p[ok]))
  wda_v <- sum(abs(a[ok]) * (sign(a[ok]) == sign(p[ok]))) / sum(abs(a[ok]))
  rows[[label]] <- data.frame(n = sum(ok), DA = da_v, WDA = wda_v)
}
period_df <- do.call(rbind, rows)
write.csv(period_df, "../../data/processed/period_midas_daily_weekly.csv", row.names = TRUE)
print(round(period_df, 3))

# Save winning variant predictions for cross-model DM tests in evaluation.ipynb
write.csv(
  data.frame(Date = dates_te, actual = y_te, predicted = best_pred),
  "../../data/processed/preds_midas_daily_best_weekly.csv",
  row.names = FALSE
)
cat(sprintf("Saved winning-variant predictions: preds_midas_daily_best_weekly.csv  (%s)\n",
            best_name))

## 11. Significance tests

Three lenses, following the `eval_utils` §3a conventions shared with the other weekly notebooks:

- **Floor test vs Drift** *(primary — the efficiency arbiter)*. Every variant vs the
  `Drift (prevailing mean)` benchmark (= ARIMA(0,0,0), the random walk with drift). Reported two
  ways: **OOS R²** (Campbell-Thompson — % MSE reduction vs the drift; effect size) and **DM**
  (significance). Squared-error DM is the headline; absolute-error DM is the heavy-tail
  robustness check. A model only beats the market if it clears this floor.
- **Incremental test vs EXOG** *(secondary)*. Each richer variant vs the `EXOG-d` base — "do
  Macro / sentiment add anything beyond cross-asset returns?" Not the efficiency claim, since
  cross-asset returns are themselves public information.
- **Pesaran-Timmermann** *(directional, secondary)*. DM is a magnitude test; PT tests whether
  the *sign* calls beat chance — the significance companion to DA/WDA. Degenerate for the
  constant-sign drift.


In [ ]:
# ── Primary floor test: each variant vs Drift (OOS R^2 + DM, squared & absolute) ──
drift_p <- all_preds[["Drift (prevailing mean)"]]
cat("Floor test — each variant vs Drift (prevailing mean)   [primary: the efficiency arbiter]\n")
cat("OOS R2 > 0 and DM < 0 => beats the random-walk drift.   * p<0.05  ** p<0.01  *** p<0.001\n")
cat(strrep("-", 92), "\n")
for (name in names(all_preds)) {
  if (name %in% c("Drift (prevailing mean)", "Naive (t-1 week)")) next
  cat(sprintf("%-26s  OOS_R2=%+.4f\n", name, oos_r2(y_te, all_preds[[name]], drift_p)))
  diebold_mariano(y_te, all_preds[[name]], drift_p, name, "Drift", loss = "se")
  diebold_mariano(y_te, all_preds[[name]], drift_p, name, "Drift", loss = "ae")
}

# ── Secondary incremental test: each richer variant vs the EXOG base ──────────
cat("\nIncremental test — each variant vs EXOG-d base   [secondary: feature value, not efficiency]\n")
cat(strrep("-", 92), "\n")
exog_p <- all_preds[["EXOG-d"]]
for (name in names(all_preds)) {
  if (name %in% c("EXOG-d", "Naive (t-1 week)", "Drift (prevailing mean)")) next
  diebold_mariano(y_te, exog_p, all_preds[[name]], "EXOG", name, loss = "se")
}

# ── Secondary directional test: Pesaran-Timmermann (sign predictability) ──────
cat("\nPesaran-Timmermann — directional (sign) predictability   [secondary]\n")
cat(strrep("-", 92), "\n")
for (name in names(all_preds)) {
  if (name == "Naive (t-1 week)") next
  pt <- pesaran_timmermann(y_te, all_preds[[name]])
  if (is.na(pt$PT)) {
    cat(sprintf("%-26s  DA=%.3f  PT degenerate (constant-sign forecast)\n", name, pt$DA))
  } else {
    sig <- if (pt$p < 0.001) "***" else if (pt$p < 0.01) "**" else if (pt$p < 0.05) "*" else "(ns)"
    cat(sprintf("%-26s  DA=%.3f  PT=%+.3f  p=%.3f  %s\n", name, pt$DA, pt$PT, pt$p, sig))
  }
}


## 12. 2026 zoom — actual vs best variant

Same 2026 view as the other weekly notebooks. The "best" variant is selected by WDA — this is
**descriptive display only**; the load-bearing efficiency verdict is the DM-vs-Drift floor test
in §10.


In [ ]:
df26 <- data.frame(date = dates_te, actual = y_te, pred = best_pred)
df26 <- df26[df26$date >= as.Date("2026-01-01"), ]

if (nrow(df26) == 0) {
  cat("No 2026 data in test set yet.\n")
} else {
  ggplot(df26, aes(x = date)) +
    geom_hline(yintercept = 0, colour = "grey60", linewidth = 0.3, linetype = "dotted") +
    geom_line(aes(y = actual), colour = "black",  linewidth = 0.8) +
    geom_point(aes(y = actual), colour = "black", size = 2) +
    geom_line(aes(y = pred),   colour = "#c0392b", linewidth = 0.8, linetype = "dashed") +
    geom_point(aes(y = pred),  colour = "#c0392b", size = 2) +
    labs(title = paste0("2026 — actual vs ", best_name, " (weekly log-returns)"),
         x = NULL, y = "Weekly log-return") +
    theme_minimal(base_size = 11)
}
